# Download system resources

Download every **file revision** at a system **branch** to your machine. Branches are snapshot tags (`baseline`, `main`, and user branches). The helper writes a **single file** when the branch has one revision, or a **`.zip` archive** when it has several.

You will:

1. Connect with `istari_labs_helpers`
2. List branches on your system
3. Download all revisions from the chosen branch

Uses [`istari_labs_helpers`](../../istari-labs-helpers) — same connect pattern as [`chaining_jobs.ipynb`](../chaining_jobs.ipynb).

### Prerequisites

- **`istari-labs-helpers`** (from cookbook root: `cd istari-labs-helpers && uv sync --extra experiment`).
- [`samples/.env`](../.env) with `ISTARI_REGISTRY_URL` and `ISTARI_PERSONAL_ACCESS_TOKEN`.
- A system id and branch name from the Istari Digital web app (for example `baseline` or `main`).

### Install kernel (optional)

From the cookbook repository root:

```bash
cd istari-labs-helpers
uv sync --extra experiment
uv run python -m ipykernel install --user --name istari-labs-helpers --display-name "Python (istari_labs_helpers)"
```

Select **Python (istari_labs_helpers)** in the kernel picker.

### Running order

Run **Connect**, edit **Prep**, then §1–§2.

## Setup

**Connect** loads credentials. **Prep** sets the system id and branch name to download.

### Connect

In [ ]:
from pathlib import Path

from istari_labs_helpers import IstariPlatform

_cwd = Path.cwd()
_env = _cwd.parent / ".env" if (_cwd.parent / ".env").exists() else _cwd / "samples" / ".env"

platform = IstariPlatform.from_env(str(_env))
print(platform)
print(f"Signed in as: {platform.whoami()}")

### Prep

Set **`SYSTEM_ID`** from the system page URL and **`BRANCH_NAME`** to the snapshot tag you want — for example `baseline` or `main`.

`DOWNLOAD_DIR` is `~/temp` (created if missing) — outside the repository so downloads do not clutter the project.

In [ ]:
SYSTEM_ID = "71a820e4-d0ea-4c35-9d3f-979bf190d410"
BRANCH_NAME = "baseline"

DOWNLOAD_DIR = Path.home() / "temp"
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

print(f"System: {SYSTEM_ID}")
print(f"Branch: {BRANCH_NAME}")
print(f"Output directory: {DOWNLOAD_DIR.resolve()}")

## 1 · Inspect branches

List snapshot tags (branches) on the system so you can confirm the branch name before downloading.

In [ ]:
system = platform.get_system_by_id(SYSTEM_ID)
print(f"System: {system.name} ({system.id})")
print("Branches:")
for branch in system.branches():
    n_revisions = len(branch.list_revisions())
    kind = "baseline" if branch.is_baseline else "branch"
    print(f"  {branch.name!r}  ({kind})  —  {n_revisions} revision(s)")

## 2 · Download resources

`download_system_resources` fetches each file revision at the branch HEAD and writes:

- **one file** when the branch has a single revision
- a **`.zip`** when it has two or more

Member names use the revision filename from the branching API.

In [ ]:
result = platform.download_system_resources(
    SYSTEM_ID,
    BRANCH_NAME,
    dest=DOWNLOAD_DIR,
)

print(f"Downloaded {result.file_count} resource(s)")
print(f"Archive type: {'zip' if result.is_zip else 'single file'}")
print(f"Path: {result.path.resolve()}")
print("Members:")
for name in result.members:
    print(f"  {name}")

### Alternative: via `SystemView`

Same download through the system object if you already have it:

```python
result = system.download_resources(BRANCH_NAME, dest=DOWNLOAD_DIR)
# or
result = system.get_branch(BRANCH_NAME).download_resources(DOWNLOAD_DIR)
```